In [1]:
import os
from re import search
from dotenv import load_dotenv
from langchain_core import embeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain.messages import HumanMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

load_dotenv()

/Users/msirdaneh/Transformation Projects/04 Training and Development/LLM_Projects/langchain-course/03_rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
print("Initializing components...")
embeddings = OpenAIEmbeddings()
llm =ChatOpenAI(model="gpt-6-luna")

vectorstore = PineconeVectorStore(
    index_name=os.environ["INDEX_NAME"],
    embedding= embeddings
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

prompt_template = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context:

{context}

Question: {question}

Provide a detailed answer:"""
)

Initializing components...


In [3]:
def format_docs(docs):
    """Format retrievded documents into a single string """
    return "\n\n".join(doc.page_content for doc in docs)

In [4]:
print("Retrieving...")
#Query
query = "what is Pinecone in machine learning"

# Option 1
print("\n" + "=" *70)
print("IMPLMENTATION 0: Raw LLM Invocation (No RAG)")
print("=" * 70)
result_raw = llm.invoke([HumanMessage(content=query)])
print("\nAnswer")
print(result_raw.content)

Retrieving...

IMPLMENTATION 0: Raw LLM Invocation (No RAG)

Answer
**Pinecone** is a managed **vector database** used in machine-learning applications to store and search embeddings—numeric representations of text, images, or other data.

An application typically:
1. Converts data into embeddings using an embedding model.
2. Stores those embeddings in Pinecone, often with metadata.
3. Searches for vectors similar to a query vector and retrieves the matching records.

This is useful for **semantic search**, recommendations, and **retrieval-augmented generation (RAG)**, where an AI model retrieves relevant information before answering a question.

Pinecone is infrastructure for finding and managing relevant data; it is **not itself a language model or an embedding model**.


In [7]:
# ============================================================================
# IMPLEMENTATION 1: Without LCEL (Simple Function-Based Approach)
# ============================================================================

def retrieval_chain_without_lcel(query:str):
    docs = retriever.invoke(query) # return list of documents, get similaries mot relevant documents
    context = format_docs(docs)
    messages = prompt_template.format(context=context, question=query)
    response = llm.invoke(messages)
    return response.content

In [8]:
# ========================================================================
# Option 1: Use implementation WITHOUT LCEL
# ========================================================================
print("\n" + "=" * 70)
print("IMPLEMENTATION 1: Without LCEL")
print("=" * 70)
result_without_lcel = retrieval_chain_without_lcel(query)
print("\nAnswer:")
print(result_without_lcel)


IMPLEMENTATION 1: Without LCEL

Answer:
Pinecone is a fully managed, cloud-based vector database used in machine-learning applications. It stores and retrieves data represented as vectors, making it possible to find data points that are similar to a query vector.

It is designed for large-scale applications, including workloads with millions or billions of data points, and supports high query throughput with low-latency search. Pinecone also supports real-time updates so newly added data can be incorporated and the database kept current.

Because Pinecone is managed, it handles infrastructure management and maintenance for its users. It provides a simple API for storing and retrieving vector data, which can make it easier to integrate into existing machine-learning workflows. The context also describes it as a secure platform for business and organizational needs, and notes that it can be synced with data from sources using tools such as Airbyte and monitored with Datadog.
